In [1]:
import embedding
from transformers import BertTokenizer
from tqdm import tqdm
import torch
import pickle

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
PTH = r'.\picklefiles'
with open(f'{PTH}\\eal_gnn_trainsmall.pkl', 'rb') as eal:
    data = pickle.load(eal)
eal.close()

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-large-uncased")
pretrained = 'bert-large-uncased'
ent_emb = embedding.EntityEmbedding(pretrained = pretrained, device = device)

Some weights of the model checkpoint at bert-large-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [39]:
assent_key = {}
for ind in tqdm(range(len(ent))):
    for el in ent[ind]['entities']:
        if el['original_id'] not in assent_key.keys():
            entity = el['entity']
            token = tokenizer.tokenize(entity)
            input_ids = tokenizer.convert_tokens_to_ids(token)
            input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
            assent_key[el['original_id']] = ent_emb(input_ids, dim = 150)

100%|██████████████████████████████████████████████████████████████████████████████| 5498/5498 [03:12<00:00, 28.62it/s]


In [40]:
with open(f'{PTH}\\associated_entemb_dict_gnn_trainsmall.pkl', 'wb') as f:
    pickle.dump(assent_key, f)
    print('Dumped')
f.close()

Dumped


In [41]:
ass_emb = torch.zeros(len(assent_key.keys()), 150)
i = 0
for el in assent_key:
    ass_emb[i] = assent_key[el]
    i += 1
    
with open(f'{PTH}\\associated_entemb_gnn_trainsmall.pkl', 'wb') as f:
    pickle.dump(ass_emb, f)
    print('Dumped')
f.close()

Dumped


In [5]:
ass_asp_key = {}
for ind in tqdm(range(len(asp))):
    tasp_id = asp[ind]['true_aspect_id']
    for el in asp[ind]['candidate_aspects']:
        asp_id = el['aspect_id']
        for ent in el['entities']:
            if ent['original_id'] not in ass_asp_key.keys():
                entity = ent['entity_name']
                token = tokenizer.tokenize(entity)
                input_ids = tokenizer.convert_tokens_to_ids(token)
                input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
                ass_asp_key[ent['original_id']] = ent_emb(input_ids, dim = 150)

100%|██████████████████████████████████████████████████████████████████████████████| 5498/5498 [22:56<00:00,  3.99it/s]


In [6]:
with open(f'{PTH}\\associated_aspent_emb_dict_gnn_trainsmall.pkl', 'wb') as f:
    pickle.dump(ass_asp_key, f)
    print('Dumped')
f.close()

Dumped


In [8]:
ass_aspemb = torch.zeros(len(ass_asp_key.keys()), 150)
i = 0
for el in ass_asp_key:
    ass_aspemb[i] = ass_asp_key[el]
    i += 1
    
with open(f'{PTH}\\associated_aspentemb_gnn_trainsmall.pkl', 'wb') as f:
    pickle.dump(ass_aspemb, f)
    print('Dumped')
f.close()

Dumped


In [9]:
ass_aspemb.shape

torch.Size([53858, 150])